In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from dataset import SoccerDataset

In [2]:
# --- 1. Load Data ---
db_path = Path("../../data/database.sqlite")
conn = sqlite3.connect(db_path.as_posix())
df = pd.read_sql_query("SELECT * FROM Player_Formation_Preprocessed", conn)
conn.close()

player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

# Extract x, y and player stats
positions = torch.tensor(df[["x", "y"]].to_numpy(), dtype=torch.float32)
players = torch.tensor(df[player_stats_cols].to_numpy(), dtype=torch.float32)

In [3]:
class PositionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(31),
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class SimpleTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_tower = PositionTower()
        self.player_tower = PlayerTower()

    def forward(self, position, player):
        pos_emb = self.position_tower(position)
        plr_emb = self.player_tower(player)

        p_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)
        u_norm = plr_emb / (torch.linalg.norm(plr_emb, dim=1, keepdim=True) + 1e-8)

        return p_norm, u_norm

In [4]:
# 1. Get unique Match IDs BEFORE any deduplication
unique_matches = df["match_api_id"].unique()

# 2. Split the MATCH IDs (80% train, 10% val, 10% test)
train_m, temp_m = train_test_split(unique_matches, test_size=0.2, random_state=42)
val_m, test_m = train_test_split(temp_m, test_size=0.5, random_state=42)

# 3. Create filtered DataFrames
# DEDUPLICATE ONLY THE TRAINING SET: This prevents "identity memorization"
# while keeping the Validation/Test sets as realistic benchmarks.
train_df = df[df["match_api_id"].isin(train_m)].drop_duplicates(
    subset=player_stats_cols
)
val_df = df[df["match_api_id"].isin(val_m)]
test_df = df[df["match_api_id"].isin(test_m)]


# 4. Define helper to extract tensors from a dataframe
def get_tensors(target_df):
    # Ensure values are float32 for PyTorch
    pos = torch.tensor(target_df[["x", "y"]].to_numpy(), dtype=torch.float32)
    plr = torch.tensor(target_df[player_stats_cols].to_numpy(), dtype=torch.float32)

    # Simple model expects 3 inputs, so we provide dummy zeros for formation
    form = torch.zeros((len(target_df), 9, 2), dtype=torch.float32)
    return form, pos, plr


# 5. Extract Tensors for each split
train_form, train_pos, train_plr = get_tensors(train_df)
val_form, val_pos, val_plr = get_tensors(val_df)
test_form, test_pos, test_plr = get_tensors(test_df)

# 6. Build Datasets
# PRO TIP: If your SoccerDataset supports it, pass noise_std=0.02 to train_dataset
# to further prevent the model from "sniping" exact stat values.
train_dataset = SoccerDataset(train_form, train_pos, train_plr)
val_dataset = SoccerDataset(val_form, val_pos, val_plr)
test_dataset = SoccerDataset(test_form, test_pos, test_plr)

# 7. Data Loaders
# Keep batch_size=64 for training to avoid "tactical collisions" (multiple similar positions)
train_loader = DataLoader(
    train_dataset, batch_size=512, shuffle=True, num_workers=4, pin_memory=True
)
# Batch size for validation can be large for speed
val_loader = DataLoader(
    val_dataset, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True
)

print(f"Split complete.")
print(f"--- Statistics ---")
print(f"Train samples (deduplicated): {len(train_dataset)} from {len(train_m)} matches")
print(f"Val samples (original):      {len(val_dataset)} from {len(val_m)} matches")
print(f"Test samples (original):     {len(test_dataset)} from {len(test_m)} matches")

Split complete.
--- Statistics ---
Train samples (deduplicated): 59711 from 17088 matches
Val samples (original):      42720 from 2136 matches
Test samples (original):     42740 from 2137 matches


In [5]:
def train_one_epoch(loader, model, optimizer, criterion, device, temp=0.07):
    model.train()
    total_loss = 0
    for pos, _, player in loader:
        pos, player = pos.to(device), player.to(device)

        p_emb, u_emb = model(pos, player)
        logits = torch.matmul(p_emb, u_emb.T) / temp
        labels = torch.arange(pos.size(0)).to(device)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(loader, model, criterion, device, temp=0.07):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for pos, _, player in loader:
            pos, player = pos.to(device), player.to(device)
            p_emb, u_emb = model(pos, player)
            logits = torch.matmul(p_emb, u_emb.T) / temp
            labels = torch.arange(pos.size(0)).to(device)
            loss = criterion(logits, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTwoTower().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

num_epochs = 200
best_val_loss = float("inf")

print(f"Starting training on {device}...")

for epoch in range(num_epochs):
    train_loss = train_one_epoch(train_loader, model, optimizer, criterion, device)
    val_loss = validate(val_loader, model, criterion, device)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "simple_soccer_model.pth")
        print(f"--> Best model saved with Val Loss: {val_loss:.4f}")

Starting training on cuda...
Epoch 1/200 | Train Loss: 5.3692 | Val Loss: 5.9036
--> Best model saved with Val Loss: 5.9036
Epoch 2/200 | Train Loss: 5.2151 | Val Loss: 5.8832
--> Best model saved with Val Loss: 5.8832
Epoch 3/200 | Train Loss: 5.1967 | Val Loss: 5.8760
--> Best model saved with Val Loss: 5.8760
Epoch 4/200 | Train Loss: 5.1870 | Val Loss: 5.8688
--> Best model saved with Val Loss: 5.8688
Epoch 5/200 | Train Loss: 5.1719 | Val Loss: 5.8590
--> Best model saved with Val Loss: 5.8590
Epoch 6/200 | Train Loss: 5.1656 | Val Loss: 5.8515
--> Best model saved with Val Loss: 5.8515
Epoch 7/200 | Train Loss: 5.1548 | Val Loss: 5.8561
Epoch 8/200 | Train Loss: 5.1423 | Val Loss: 5.8444
--> Best model saved with Val Loss: 5.8444
Epoch 9/200 | Train Loss: 5.1336 | Val Loss: 5.8426
--> Best model saved with Val Loss: 5.8426
Epoch 10/200 | Train Loss: 5.1218 | Val Loss: 5.8289
--> Best model saved with Val Loss: 5.8289
Epoch 11/200 | Train Loss: 5.1109 | Val Loss: 5.8330
Epoch 12/2

KeyboardInterrupt: 